In [4]:
#!/usr/bin/env python
# coding: utf-8

########################################ALL IMPORTS ############################################
import pandas as pd
import numpy as np
import random
import itertools
import json
import pprint
import datetime
from datetime import date, time, timedelta, datetime
import requests
import os
import pandas_market_calendars as mcal
import base64
from datetime import datetime, timedelta, date
import traceback
import glob
import sys

# Add the parent directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from functions import *
from config import appKey, appSecret

# DIFFERENT THINGS TO TEST #############
# Now we tweak to incorporate different signals to improve odds.
    
# 2) INCORPORATE THE DIFFERENT SIGNALS AND TRY THE THINGS BELOW:
    
# I could probably try all of these in another for loop of 5 binary variables and then encode them into the signals portion of the code. Just like how  i tested all the strategies, i will need to write code to test all the different signals.

# 1) 10m bars, 15m bars, 1 hr?
# 2) Spike must be higher than both any other open, or any other low, or any other high?
# 3) INCORPORATE VOLATILITY, implied vol discount, momentum, other market factors
# 4) Create market cap, float buckets, pre-market volume, and bar volume to see patterns emerge and perform backwards      looking analysis
# 5) TRY DIFFERENT TARGET ENTRIES (VWAP, VWAP_STD, VWAP*95%
# 6) PREMARKET HIGH no buy?
# 7) FULL GREEN BAR (MEASURE TO CLOSE MUST BE HIGHER THAN ALL VALUES OF DAY)(PROB NOT)
# 8) THE HIGH OF VOLUME SPIKE MUST BE HIGHER THAN ALL THE OTHER CLOSES OF THE DAY
# 9) DIFFERENT EXITS (stop and target to be set based on volatility of stock? Or on rolling volatility?
# 10) TIMING THRESHOLDS: 1) vol spike signal before x 2) buy before x 3) sell before x
# 11) BACKTEST ENHANCEMENTS FOR FUTURE STRATEGIES

# 12)DON'T BUY IF ACCOUNT_SIZE IS TOO SMALL TO BUY
# 13)CREATE IN-HOUSE FUNCTIONS


In [6]:
## DEFINING ALL VARIABLES AND THINGS POSSIBLE TO TOGGLE FOR BACKTESTING

######################################## STRATEGY NOTE TO REMIND ON OUTPUT ########################################
strategy_note = 'quick_test'
run = strategy_note

########################################  IMPORT LIST OF ALL TICKERS FOR BACKTEST/STRATEGY   #################################
#Occasionally should update this list. Here is the criteria for the csv
# filtered_df = df[(df['Exchange'].isin(['NASD', 'NYSE'])) & 
#                  (df['Market Cap'] > 0) & 
#                  (df['Float'] > 0)]

ticker_list = pd.concat(map(pd.read_excel, glob.glob("../data/by_float/*.xlsx")))
ticker_list = ticker_list[ticker_list['Float'] > 50000000]

########################################  DEBUGGING - TEST FOR SINGLE TICKER   #################################
# For initial testing, we'll just use a single ticker
# tickers = ['IONQ']  # Can easily change this to test different tickers
# print(f"Testing strategy on ticker: {tickers[0]}")

####################################### SETTING CANDLE TIME FRAME FOR STRATEGY ########################################

# Define variables for stock time frame for strategy
period_type = 'day'  # Current value
period = 10          # Current value
frequency_type = 'minute'  # Current value
frequency = 30      # Current value
need_extended_hours_data = 'true'  # Current value
need_previous_close = 'true'  # Current value

######################################## TESTING COMBINATION OF INPUTS FOR STRATEGY  ########################################

#THIS IS THE STRATEGY I DECIDED ON:

# TOTAL_CASH = 100000 #FIXED
# BET_SIZE = [.1] #FIXED
# STOP = [.1] #FIXED #Removed .05
# TARGET = [.05] #FIXED

# VOL_SPIKE_THRESHOLD = [5] #Abnormally high volume that stands out on a chart
# PRICE_SPIKE_THRESHOLD = [.05] #Must move the price x%
# TIME_SIG_THRESHOLD = [time(hour=12,minute=30,second=0)]
# BUY_TIME_THRESHOLD = [ time(hour=10,minute=30,second=0)]
# SELL_TIME_THRESHOLD = [time(hour = 15,minute = 30,second =0)]

#TESTING MULTIPLE VARIATIONS AT ONCE

TOTAL_CASH = 100000 #FIXED
BET_SIZE = [.1] #FIXED
STOP = [.1,.15,1] #FIXED #Removed .05
TARGET = [.05,.1] #FIXED

VOL_SPIKE_THRESHOLD = [5,10] #Abnormally high volume that stands out on a chart
PRICE_SPIKE_THRESHOLD = [.05,.1] #Must move the price x%
TIME_SIG_THRESHOLD = [time(hour=12,minute=30,second=0)]
BUY_TIME_THRESHOLD = [ time(hour=10,minute=30,second=0)]
SELL_TIME_THRESHOLD = [time(hour = 15,minute = 30,second =0)]

####################################### CREATE VARIABLES FOR INPUT STRATEGY TO TEST ##########################

bet_size_index = 0
stop_index = 1
target_index = 2
vol_spike_thresh_index = 3
price_spike_thresh_index = 4
time_sig_thresh_index = 5
buy_time_threshold_index = 6
sell_time_threshold_index = 7

variables = [BET_SIZE,STOP,TARGET,VOL_SPIKE_THRESHOLD,PRICE_SPIKE_THRESHOLD,TIME_SIG_THRESHOLD,BUY_TIME_THRESHOLD, SELL_TIME_THRESHOLD]
combinations = list(itertools.product(*variables))      

######################################## STRATEGY OUTPUT TABLE ########################################

#FINAL OUTPUT TO EVALUATE DIFFERENT STRATEGY COMBINATIONS

inputs = pd.DataFrame(columns=['Account Size',
                               'Bet_Size',
                               'Stop',
                               'Target',
                               'Vol Spike Thresh',
                               'Price Spike Thresh',
                               'Signal Time Threshold',
                               'Buy Time Threshold',
                               'Sell_Time',
                               'Signals',
                               'Buys',
                               'Win %',
                               'Average Win',
                               'Strategy Note'])

########################################  TIME FRAME  ########################################

# Set date parameters and calculations regarding dates such as 10 day rolling average volume

en = datetime.now()
st = en - timedelta(days=20)

days = (en-st).days

start_time = str(int(st.timestamp())*1000)
end_time = str(int(en.timestamp())*1000)

today = date.today()

time_interval = 30
time_interval_string = str(time_interval)+'m'
rolling_window_days = 10
tickers_per_day = 60/time_interval*6.5
rolling_lookback = rolling_window_days*tickers_per_day

# Find Next Business Day
# Import the New York Stock Exchange Calendar
nyse = mcal.get_calendar('NYSE')
open_close_schedule = pd.DataFrame(nyse.schedule(start_date=st, end_date=en))
open_close_schedule.index.names = ['Date']
open_close_schedule.reset_index(inplace=True)
open_close_schedule['Date'] = open_close_schedule['Date'].dt.date
open_close_schedule['market_open'] = open_close_schedule['market_open'] - timedelta(hours=4)
open_close_schedule['market_close'] = open_close_schedule['market_close'] - timedelta(hours=4, minutes=time_interval)
open_close_schedule['market_open'] = open_close_schedule['market_open'].dt.time
open_close_schedule['market_close'] = open_close_schedule['market_close'].dt.time

nyse_days = nyse.valid_days(start_date=st, end_date=en)
valid_trading_days = pd.Series(nyse_days).dt.date

schedule = pd.DataFrame(nyse.schedule(start_date=st, end_date=en))


In [144]:
# Download data into memory
# Ensure the 'Tickers' column exists
if 'Ticker' in ticker_list.columns:
    # Create a new DataFrame with only the 'Tickers' column
    tickers_df = ticker_list[['Ticker']].dropna()
    # Convert to a list of unique tickers
    tickers = tickers_df['Ticker'].unique().tolist()  # Extract unique tickers
    print("First few tickers:", tickers[:5]) 
    print('SUCESS IF YOU SEE TICKERS!!! ^^^^^^^')
else:
    print("Column 'Ticker' not found in the DataFrame.")

First few tickers: ['BL', 'ALKT', 'STEW', 'CRL', 'ZBRA']
SUCESS IF YOU SEE TICKERS!!! ^^^^^^^


In [ ]:
# tickers = ['IONQ']  # Can easily change this to test different tickers
# print(f"Testing strategy on ticker: {tickers[0]}")

In [7]:
################################## TESTING/AUTHENTICATING CONNECTION TO SCHWAB API ##################################
# Get access token (figure out how often to do this optimally)
access_token = auto_authenticate(appKey, appSecret)

# Get AAPL price history for the last year using the defined variables
aapl_price_history = get_stock_price_history('AAPL', access_token, period_type, period, frequency_type, frequency,  start_time, end_time,need_extended_hours_data, need_previous_close)

# Convert to DataFrame
aapl_df = pd.DataFrame(aapl_price_history['candles'])

# Rename columns to match the required format
aapl_df.rename(columns={
    'datetime': 'Datetime',
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'close': 'Close',
    'volume': 'Volume'
}, inplace=True)

# Convert Datetime to EST
aapl_df['Datetime'] = pd.to_datetime(aapl_df['Datetime'], unit='ms').dt.tz_localize('UTC').dt.tz_convert('US/Eastern').dt.tz_localize(None)

# Save the DataFrame as a CSV file
# aapl_df.to_csv('./file_uploads/tests/aapl_price_history.csv', index=False)

# Print the result
print(aapl_df.head()) 
print('SUCCESS IF I SEE APPLE STOCK DATA!!!')


       Open    High     Low    Close  Volume            Datetime
0  233.6900  234.23  233.69  234.190    9625 2024-10-16 07:00:00
1  234.2000  234.22  233.85  233.958   11820 2024-10-16 07:30:00
2  234.0400  234.04  232.84  233.000   79014 2024-10-16 08:00:00
3  232.9513  233.02  232.84  232.910   46418 2024-10-16 08:30:00
4  232.9100  233.23  231.40  231.700  242058 2024-10-16 09:00:00
SUCCESS IF I SEE APPLE STOCK DATA!!!


In [131]:

######################### PULL INITIAL DATAFRAME OF STOCK PRICE HISTORY AND PERFORM PRECALCULATIONS #########################

start_clock = datetime.now()  # calculate run time
go = 1
stockies = {} #Create dataframes of stock data for iteration

for ticker in tickers:
    # Get stock price history from Schwab API
    stock_data = get_stock_price_history(ticker, access_token, period_type, period, frequency_type, frequency, start_time, end_time, need_extended_hours_data, need_previous_close)
    
    if not stock_data or 'candles' not in stock_data:
        print(f"No data available for {ticker}")
        continue
    
    # Convert to DataFrame
    stahks = pd.DataFrame(stock_data['candles'])
    
    # Ensure all required fields are present
    required_fields = ['datetime', 'open', 'high', 'low', 'close', 'volume']
    if not all(field in stahks.columns for field in required_fields):
        print(f"Missing required fields for {ticker}. Available columns: {stahks.columns}")
        continue
    
    # Rename columns to match the required format
    stahks.rename(columns={
        'datetime': 'Datetime',
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close',
        'volume': 'Volume'
    }, inplace=True)
    
    # Convert Datetime to EST
    stahks['Datetime'] = pd.to_datetime(stahks['Datetime'], unit='ms').dt.tz_localize('UTC').dt.tz_convert('US/Eastern').dt.tz_localize(None)
    
    # Print the DataFrame to inspect its structure
    # print(f"Data for {ticker}:")
    # print(stahks.head())  # Display the first few rows of the DataFrame
    # print("Columns in DataFrame:", stahks.columns)  # Print the column names
    
    #Skip stock if there is insufficient amount of data (data for each period of normal trading hours)
    end_check = stahks['Datetime'].max()
    start_check = stahks['Datetime'].min()
    daydiff = end_check.weekday() - start_check.weekday()
    days = ((end_check-start_check).days - daydiff) / 7 * 5 + min(daydiff,5) - (max(end_check.weekday() - 4, 0) % 5)
    
    print(ticker, len(stahks.index))    
    # Removed the erroneous line that referenced 'datetime' instead of 'Datetime'
    # stahks['datetime'] = pd.to_datetime(stahks['datetime']/1000, unit = 's')-timedelta(hours =4)
    # stahks.columns = ['Open','High','Low','Close','Volume','Datetime']
    
    #Rearrange Columns and Merge with Open/Close Schedule
    stahks['Ticker'] = ticker
    stahks['Date'] = stahks['Datetime'].dt.date
    stahks = stahks.merge(open_close_schedule,how = 'left', on = 'Date')

    # Only calculate and average volume if there is sufficient data
    rolling_lookback_int = int(rolling_lookback)
    stahks['10_Day_Avg_Vol'] = stahks.Volume.rolling(rolling_lookback_int, min_periods=rolling_lookback_int).mean()
    stahks['10_Day_Avg_Vol'] = stahks['10_Day_Avg_Vol'].fillna(float('inf'))
    stahks['Time'] = stahks['Datetime'].dt.time
    
    #Remove any accidental duplicates (FIGURE OUT WHY????)
    stahks.drop_duplicates(['Ticker','Date','Time'],inplace = True,ignore_index=True)
    
    if len(stahks.index) < (days * tickers_per_day):
        continue
        
    # Create column with the open bar's low price (For % gap up calculation with spike later in day)
    cond = (stahks['Time'] == stahks['market_open'])
    stahks['Day_Open_Low'] = stahks[cond].groupby('Date', as_index=True)['Low'].transform('min').ffill()
    
    stahks['After Hours'] = (stahks['Time'] > stahks['market_close']) | (stahks['Time'] < stahks['market_open'])

    cond_2 = (stahks['After Hours'] == True)
    stahks['Pre-Market High'] = stahks[cond_2].groupby('Date', as_index=True)['High'].transform('max')
    
    stahks = stahks.ffill(axis=0)
    stahks = stahks.bfill(axis=0)

    
    stahks['VWAP_Row'] = stahks['Volume']*((stahks['High']+stahks['Low']+stahks['Close'])/3)
    stahks['Cum_VWAP'] = stahks.groupby('Date')['VWAP_Row'].transform('cumsum')
    stahks['Cum_Volume'] = stahks.groupby('Date')['Volume'].transform('cumsum')
    stahks['VWAP'] = stahks['Cum_VWAP']/stahks['Cum_Volume']
    stahks['VWAP_STD_1'] = stahks['VWAP'] - stahks.groupby('Date')['VWAP'].transform('std')
    stahks['Color_Bar'] = np.where(stahks['Open']<=stahks['Close'], 'Green', 'Red')
    #vol_window = 1
    stahks['Day_Close'] = (stahks['Time'] == stahks['market_close'])

    
    stockies[ticker] = pd.DataFrame(stahks, columns=stahks.keys())
    print(f"{ticker} processed successfully.")
    go += 1

    #Calculate pre-market volume for day 
    #Calculate pre-market change for the day 
    #stockies['Ticker_Return'] = (stahks['Close']/stahks['Close'].shift(vol_window))-1
    #stockies['Rolling_Vol'] = stahks['Ticker_Return'].std(ddof=130)
    
    ##PRINT OUT THE DATAFRAME TO A CSV FILE
    # stahks.to_csv(f"backtest_pre.csv", index=False, header=True)

#Stockies is the final dataframe that will be used for the backtest




IONQ 467
IONQ processed successfully.


In [132]:
######################### START OF BACKTESTING ########################################################

input_indexer = 1

for strategy in combinations:
    #Skips combos where the buy time threshold is after the sell time threshold
    if (strategy[buy_time_threshold_index]>strategy[sell_time_threshold_index]):
        continue           
    
    print(input_indexer, datetime.now())
    
########################################### DEFINING BACKTEST RESULTS TABLE FOR EACH VARIATION OF STRATEGY ########################################

    results = pd.DataFrame(columns=['Ticker',
                                    'Date',
                                    'Volume Spike',
                                    'Price Spike',
                                    'Previous Day Close',
                                    'Signal Time',
                                    'Target Entry',
                                    'Entry Time',
                                    'Exit Time',
                                    'Account Size',
                                    'Bet Size',
                                    'Win/Loss', 
                                    'Profit',
                                    'Profit %',
                                    'Entry',
                                    'Exit',
                                    'Type',
                                    'Volume',
                                    'Premarket Volume',
                                    'Premarket Change'])
                                    
    RESULT_INDEXER= 0
    #Create temporary variable to output total number of signals and purchases found (where we'd need to be prepared to buy on the next day)
    SIGNALS = 0
    BUYS = 0
    ACCOUNT_SIZE = TOTAL_CASH

    ################################################   IMPORT DATA   ##################################################         
    
    tickers = list(stockies.keys())
    random.shuffle(tickers)
    
    for ticker in tickers:
    ###########################################  CALCULATE SIGNALS FOR BACKTEST BASED ON STRATEGY INPUTS  ###########################################

        #Checks for Highest Daily Volume
        high_vol_sig = np.where(stockies[ticker]['Volume'] == stockies[ticker].groupby('Date')['Volume'].transform('max'),'True','False')
        stockies[ticker]['high_vol_sig'] = high_vol_sig
        #Checks for price spike more than X% greater than 
        price_sig = np.where((stockies[ticker]['High']-stockies[ticker]['Day_Open_Low'])/stockies[ticker]['Day_Open_Low'] >= strategy[price_spike_thresh_index],'True','False')
        stockies[ticker]['price_sig'] = price_sig
        #Checks that the spike was before X time threshold (tied to highest daily volume)
        before_time_thresh = np.where(stockies[ticker]['Time'] <= strategy[time_sig_thresh_index], 'True','False')
        stockies[ticker]['before_time_thresh'] = before_time_thresh
        #Checks that spike was a green bar
        green_bar = np.where(stockies[ticker]['Color_Bar'] == 'Green', 'True','False')
        stockies[ticker]['green_bar'] = green_bar
        #Checks for volume spike X% greater than 10day average
        vol_spike_sig = np.where(stockies[ticker]['Volume'] > strategy[vol_spike_thresh_index] * stockies[ticker]['10_Day_Avg_Vol'],'True','False')
        stockies[ticker]['vol_spike_sig'] = vol_spike_sig
        #Checks that the spike high was the highest price of the day
        high_price_sig = np.where(stockies[ticker]['High'] >= stockies[ticker].groupby('Date')['Close'].transform('max'),'True','False')
        stockies[ticker]['high_price_sig'] = high_price_sig
        #Checks to see if it time is before equal to the sell_time threshold set by user
        stockies[ticker]['sell_time'] = np.where(stockies[ticker]['Time'] == strategy[sell_time_threshold_index],'True','False')
        #Finds High of the Day
        stockies[ticker]['high_of_day'] = stockies[ticker].groupby('Date')['High'].transform('max')

        #Checks all conditions
        stockies[ticker]['Grab_Price_Signal'] = np.where((high_vol_sig == 'True') 
                                                         & (vol_spike_sig == 'True') 
                                                         & (price_sig == 'True') 
                                                         & (high_price_sig == 'True') 
                                                         #& (green_bar == 'True') 
                                                         & (before_time_thresh == 'True'),
                                                         'True','False')
        
    ################################ INPUT BUY AND SELL SIGNALS FOR BACKTEST ##################################################

        #Find a way to turn on and off signals and rules to be 'True' 'False' 'Ignore' - yet still works with backtest framework

        #Checks that the close was lower than the VWAP (maybe in backtest)
        stockies[ticker]['Close_Condition'] = 'False'
        
        #Checks that all criteria was checked (X days ago) and that we are just waiting for buy signal (price crosses above VWAP)
        stockies[ticker]['Ok_To_Buy'] = 'False'
        #Creates the Buy and Sell Signal Column for iteration
        stockies[ticker]['Buy_Sell_Signal'] = 'None'
        #Create target entry price column
        stockies[ticker]['Target_Entry_Price'] = 100000.0

        #Temp Variables for backtest (row by row iteration)
        #Create temporary signal day variable that signaled whether or not the price_to_buy_signal was triggered during that day
        TEMP_SIGNAL_DAY = stockies[ticker]['Date'][0] - timedelta(days=1)
        #Create temporary ok to buy day variable that will trigger if the close condition for the day was satisfied (which only triggers if grab_price_signal is triggered)
        OK_TO_BUY_DAY = stockies[ticker]['Date'][0] - timedelta(days=2)
        #Initially set not to trigger and gets set on price_buy_signal
        TARGET_ENTRY_PRICE = 0.0 
        ENTRY_PRICE = 0.0 
        ENTRY_TIME = stockies[ticker]['Time'][0]
        BOUGHT_TODAY = stockies[ticker]['Date'][0] - timedelta(days=1)
        PREVIOUS_DAY_CLOSE = 0.0
        SIGNAL_TIME = stockies[ticker]['Time'][0]
        OPTIMAL_ENTRY = 0.0
        OPTIMAL_EXIT = 0.0
        VOLUME_SPIKE = 0.0
        PRICE_SPIKE_TEMP = 0.0
        YESTERDAY_HIGH = 0.0

        OPTIMAL_ENTRY_TIME = 0.0
        OPTIMAL_EXIT_TIME = 0.0

        POSITION = 'Neutral'

        # Save stockies to see its structure
        # stockies[ticker].to_excel('./test/stockies_structure.xlsx', index=True, header=True)

        # test = f'./test/backtest.xlsx'
        # stockies[ticker].to_excel(test, index=False, header=True)

        #Iterate over rows to see which rows meet the close condition and the all clear to buy signal (pending final signal: price cross)
        #Unique to this strategy's backtest. Could be a part of inserting variables and signals before BACKTEST SECTION

        for index, row in stockies[ticker].iterrows():

            if row['Grab_Price_Signal'] == 'True':
                # print(f"Found signal: {row['Date']}")
                TEMP_SIGNAL_DAY = row['Date']
                TARGET_ENTRY_PRICE = row['VWAP']
                SIGNAL_TIME = row['Time']
                VOLUME_SPIKE = row['Volume']/row['10_Day_Avg_Vol']
                PRICE_SPIKE = (row['High']-row['Day_Open_Low'])/row['Day_Open_Low']
                YESTERDAY_HIGH = row['high_of_day']

            if (row['Close']<=TARGET_ENTRY_PRICE) & (row['Day_Close'] == True) & (row['Date'] == TEMP_SIGNAL_DAY):
                # print(f"Setting Close_Condition: Close={row['Close']}, TARGET={TARGET_ENTRY_PRICE}, Date={row['Date']}")
                stockies[ticker].at[index,'Close_Condition'] = 'True'
                OK_TO_BUY_DAY = next_business_day(row['Date'])
                SIGNALS += 1
                PREVIOUS_DAY_CLOSE = row['Close']
            
            if (OK_TO_BUY_DAY == row['Date']):
                # print(f"Setting Ok_To_Buy for date: {row['Date']}")
                stockies[ticker].at[index,'Ok_To_Buy'] = 'True' 
                stockies[ticker].at[index,'Previous_Day_Close'] = PREVIOUS_DAY_CLOSE
                stockies[ticker].at[index,'Signal Time'] = SIGNAL_TIME
                stockies[ticker].at[index,'Volume Spike'] = VOLUME_SPIKE
                stockies[ticker].at[index,'Price_Spike_From_Open'] = PRICE_SPIKE
                stockies[ticker].at[index,'Yesterday High'] = YESTERDAY_HIGH
                stockies[ticker].at[index,'Target_Entry_Price'] = TARGET_ENTRY_PRICE

        # test = f'./test/backtest.xlsx'
        # stockies[ticker].to_excel(test, index=False, header=True)
        
        #Consolidate tables to only days where we might buy and sell
        stonks = stockies[ticker][(stockies[ticker]['Ok_To_Buy'] == 'True')]
        # stonks.to_excel(f'./test/BACKTEST_STONKS_{ticker}_{input_indexer}.xlsx', index=True, header=True)

            
        #Check that pre-market high wasn't higher than yesterday's high bar (in backtest)
        #Use variable for yesterday's high that switches on signal to equal high price of signal
        #Compare to yesterday's high bar       
        #Check that pre-market high wasn't higher than yesterday's high bar (in backtest)
        #Use variable for yesterday's high that switches on signal to equal high price of signal
        #Compare to yesterday's high bar

        #If there are no signals - skip to next stock.
        if len(stockies[ticker][stockies[ticker]['Ok_To_Buy']== 'True']) == 0:
            continue      
            
        # After the loop
        # if stockies[ticker]['Ok_To_Buy'].any():
        #     print(f"Found Ok_To_Buy signals for dates:")
        #     print(stockies[ticker][stockies[ticker]['Ok_To_Buy'] == 'True'][['Date', 'Grab_Price_Signal', 'Close_Condition']])
    #########################################  BACKTEST IMPLEMENTATION AND SIMULATION OF BUY AND SELL SIGNALS  ##################################################
        
        POSITION = 'Neutral'
        
        #USING IN-HOUSE FUNCTION SIMULATE (ITERATING THROUGH DAYS, BUYING AND SELLING BASED ON SIGNALS)
        #Iterate over rows to fill in results table with buy and sell actions
        for index, row in stonks.iterrows():    
            signal = buy_sell_signal('Short',
                                     row['Ok_To_Buy'],
                                     row['Time'],
                                     row['High'],
                                     row['Low'],
                                     row['After Hours'],
                                     row['Day_Close'],
                                     row['Date'],
                                     strategy[stop_index],
                                     strategy[target_index],
                                     strategy[sell_time_threshold_index],
                                     strategy[buy_time_threshold_index],
                                     row['Yesterday High'],
                                     row['Pre-Market High'],
                                     row['Target_Entry_Price'])
            if signal == 'none':
                continue
            position_size = ACCOUNT_SIZE * strategy[bet_size_index]    
            buy_sell(signal,
                     row['Date'],
                     row['Ticker'],
                     row['Open'],
                     row['Close'],
                     row['Time'],
                     row['Volume'],
                     row['Previous_Day_Close'],
                     row['Volume Spike'],
                     row['Price_Spike_From_Open'],
                     row['Target_Entry_Price'],
                     position_size,
                     strategy[stop_index],
                     strategy[target_index])
            
            signal = buy_sell_signal('Short',
                                     row['Ok_To_Buy'],
                                     row['Time'],
                                     row['High'],
                                     row['Low'],
                                     row['After Hours'],
                                     row['Day_Close'],
                                     row['Date'],
                                     strategy[stop_index],
                                     strategy[target_index],
                                     strategy[sell_time_threshold_index],
                                     strategy[buy_time_threshold_index],
                                     row['Yesterday High'],
                                     row['Pre-Market High'],
                                     row['Target_Entry_Price'])
            if signal == 'none':
                continue
            position_size = ACCOUNT_SIZE * strategy[bet_size_index]    
            buy_sell(signal,
                     row['Date'],
                     row['Ticker'],
                     row['Open'],
                     row['Close'],
                     row['Time'],
                     row['Volume'],
                     row['Previous_Day_Close'],
                     row['Volume Spike'],
                     row['Price_Spike_From_Open'],
                     row['Target_Entry_Price'],
                     position_size,
                     strategy[stop_index],
                     strategy[target_index])
            
        #HOW DO I CHECK FOR BUY AND SELL IN THE SAME STRATEGY (JUST COPY PASTE IT AND DO IT TWICE)
        #results['Optimal Entry'] = stockies[results['Date']==stockies['Date']].groupby('Date')['High'].transform('max')
        #results['Optimal Exit'] = stockies[results['Date']==stockies['Date']].groupby('Date')['Low'].transform('min')
        #results['Optimal Entry Time'] =stockies[results['Date']==stockies['Date']].groupby('Date')['High'].transform('max')
        #results['Optimal Exit Time'] =
        #results['Left On Table'] = ((results['Optimal Entry']*results['Bet Size'])-(results['Optimal Exit']*results['Bet Size']))-((results['Bet Size'])-(STOP*results['Bet Size']))
    
    inputs.at[input_indexer,'Account Size'] = ACCOUNT_SIZE
    inputs.at[input_indexer,'Bet_Size'] = strategy[bet_size_index]
    inputs.at[input_indexer,'Stop'] = strategy[stop_index]
    inputs.at[input_indexer,'Target'] = strategy[target_index]
    inputs.at[input_indexer,'Vol Spike Thresh'] = strategy[vol_spike_thresh_index]
    inputs.at[input_indexer,'Price Spike Thresh'] = strategy[price_spike_thresh_index]
    inputs.at[input_indexer,'Signal Time Threshold'] = strategy[time_sig_thresh_index]
    inputs.at[input_indexer,'Buy Time Threshold'] = strategy[buy_time_threshold_index]
    inputs.at[input_indexer,'Signals'] = SIGNALS
    inputs.at[input_indexer,'Buys'] = BUYS
    inputs.at[input_indexer,'Average Win'] = results['Profit %'].mean()
    inputs.at[input_indexer,'#Hit Target'] = len(results[results['Type'] == 'Sell (Target Hit)'])
    inputs.at[input_indexer,'#Hit Stop'] = len(results[results['Type'] == 'Sell (Stop Loss)'])
    inputs.at[input_indexer,'#Sold at Time']= len(results[results['Type'] == 'Sell (At Time Threshold)'])
    inputs.at[input_indexer,'Sell_Time']= strategy[sell_time_threshold_index]
    inputs.at[input_indexer,'Strategy Note']= strategy_note
    try:
        inputs.at[input_indexer,'Win %'] = (len(results[results['Win/Loss']=='Win'])/len(results['Win/Loss']))
    except:
        pass
    tick_list = tickers
    # Write each dataframe to a different worksheet.
    # results = pd.merge(results,tick_list[['Ticker','Market Capitalization','Sector','Shares Float']],on = 'Ticker', how = 'left')
    results.to_excel(f'./backtest_results/detailed_results/run_{run}_strategy_{input_indexer}.xlsx', index=False, header=True)
    input_indexer += 1

################################################### CLEAN OUTPUTS ####################################################

    print ("Ending Account Size: ", ACCOUNT_SIZE)
    print ("Signals: ", SIGNALS)
    print ("Buys:", BUYS)

    try:
        print ("Win %: ", len(results[results['Win/Loss']=='Win'])/len(results['Win/Loss'])*100,"%")
        print("Avg. Win: ", results['Profit %'].mean()*100,"%")

    except:
        pass

# plot = px.line(results, x = results.index.values, y = 'Account Size', title = 'Equity Curve')
# plot.show()
current_date = datetime.now().strftime("%Y-%m-%d")
inputs.to_excel(f'./backtest_results/run_{run}_on_{current_date}.xlsx', index=True, header=True)
print(datetime.now() - start_clock)


1 2024-11-01 13:15:09.096018
Ending Account Size:  99513.020618646
Signals:  3
Buys: 2
Win %:  0.0 %
Avg. Win:  -2.4378677540403855 %
0:00:00.419494
